In [1]:
import pandas as pd

internships = pd.read_csv("/content/internships_master.csv")
students = pd.read_csv("/content/student_dataset.csv")

print("Internships:", internships.shape)
print("Students:", students.shape)

Internships: (1000, 12)
Students: (149, 9)


In [2]:
student = {
    "branch": "cse",
    "cgpa": 8.15,
    "skills": ["python", "sql", "pandas"],
    "interests": ["machine learning", "data science"],
    "career_goal": "machine learning engineer",
    "preferred_location": "remote",
    "preferred_duration": 3,
    "minimum_stipend": 10000
}

print(student)

{'branch': 'cse', 'cgpa': 8.15, 'skills': ['python', 'sql', 'pandas'], 'interests': ['machine learning', 'data science'], 'career_goal': 'machine learning engineer', 'preferred_location': 'remote', 'preferred_duration': 3, 'minimum_stipend': 10000}


In [3]:
import ast

def normalize_list(x):
    if isinstance(x, list):
        if len(x) == 1 and isinstance(x[0], str) and x[0].startswith("["):
            return [item.strip().lower() for item in ast.literal_eval(x[0])]
        return [item.strip().lower() for item in x]

    if isinstance(x, str):
        x = x.strip()

        if x.startswith("["):
            return [item.strip().lower() for item in ast.literal_eval(x)]

        return [item.strip().lower() for item in x.split("|")]

    return []


# Internship data
internships["skills"] = internships["skills"].apply(normalize_list)
internships["eligible_branches"] = internships["eligible_branches"].apply(normalize_list)

internships["location"] = internships["location"].apply(
    lambda x: [
        city.strip().lower()
        for city in str(x).replace(" (hybrid)", "").split("/")
    ]
)

internships["title"] = internships["title"].str.strip().str.lower()


# Student data
students["skills"] = students["skills"].apply(normalize_list)
students["interests"] = students["interests"].apply(normalize_list)

students["branch"] = students["branch"].str.strip().str.lower()
students["career_goal"] = students["career_goal"].str.strip().str.lower()
students["preferred_location"] = students["preferred_location"].str.strip().str.lower()

In [4]:
def get_eligible_internships(student, internships):
    return internships[
        internships["eligible_branches"].apply(
            lambda branches: student["branch"] in branches
        )
        & (internships["min_cgpa"] <= student["cgpa"])
    ].copy()



In [5]:
def skill_score(student_skills, internship_skills):
    student_skills = set(student_skills)
    internship_skills = set(internship_skills)

    if not student_skills:
        return 0.0

    return len(student_skills & internship_skills) / len(student_skills)

In [6]:
def duration_score(preferred_duration, internship_duration):
    if preferred_duration is None:
        return 0.0

    if internship_duration == preferred_duration:
        return 1.0

    if abs(internship_duration - preferred_duration) == 1:
        return 0.5

    return 0.0

In [7]:
def get_skill_matches(student, internships):
    eligible = get_eligible_internships(student, internships)

    eligible["skill_score"] = eligible["skills"].apply(
        lambda internship_skills: skill_score(
            student["skills"],
            internship_skills
        )
    )

    return eligible

In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

internships["search_text"] = (
    internships["title"] + " "
    + internships["skills"].apply(lambda x: " ".join(x)) + " "
    + internships["description"].fillna("").str.lower()
)

vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2)
)

internship_text_vectors = vectorizer.fit_transform(
    internships["search_text"]
)

In [9]:
def get_career_goal_scores(career_goal):
    student_vector = vectorizer.transform([career_goal])

    return cosine_similarity(
        student_vector,
        internship_text_vectors
    ).flatten()

In [10]:
def get_interest_scores(interests):
    if not interests:
        return [0.0] * len(internships)

    interest_vectors = vectorizer.transform(interests)

    similarities = cosine_similarity(
        interest_vectors,
        internship_text_vectors
    )

    return similarities.mean(axis=0)

In [11]:
def location_score(preferred_location, internship_locations):
    preferred_location = preferred_location.strip().lower()

    if "remote" in internship_locations:
        return 1.0

    if preferred_location in internship_locations:
        return 1.0

    return 0.0

In [12]:
def stipend_score(minimum_stipend, internship_stipend):
    if internship_stipend >= minimum_stipend:
        return 1.0

    return 0.0

In [13]:
def get_candidate_internships(student):
    eligible = get_eligible_internships(student, internships)

    # Career goal
    career_scores = get_career_goal_scores(student["career_goal"])
    eligible["career_goal_score"] = [
        career_scores[idx] for idx in eligible.index
    ]

    # Skills
    eligible["skill_score"] = eligible["skills"].apply(
        lambda skills: skill_score(student["skills"], skills)
    )

    # Interests
    interest_scores = get_interest_scores(student["interests"])
    eligible["interest_score"] = [
        interest_scores[idx] for idx in eligible.index
    ]

    # Location
    eligible["location_score"] = eligible["location"].apply(
        lambda locations: location_score(
            student["preferred_location"],
            locations
        )
    )

    # Duration
    eligible["duration_score"] = eligible["duration_months"].apply(
        lambda duration: duration_score(
            student["preferred_duration"],
            duration
        )
    )

    # Stipend
    eligible["stipend_score"] = eligible["stipend"].apply(
        lambda stipend: stipend_score(
            student["minimum_stipend"],
            stipend
        )
    )

    return eligible

def calculate_final_score(df):
    df["final_score"] = (
        0.35 * df["skill_score"]
        + 0.25 * df["career_goal_score"]
        + 0.15 * df["interest_score"]
        + 0.10 * df["location_score"]
        + 0.075 * df["duration_score"]
        + 0.075 * df["stipend_score"]
    )

    return df

def recommend_internships(student, top_n=10):
    candidates = get_candidate_internships(student)
    candidates = calculate_final_score(candidates)

    recommendations = candidates.sort_values(
        "final_score",
        ascending=False
    ).head(top_n)

    return recommendations

In [14]:
recommendations = recommend_internships(
    student,
    top_n=10
)

print(
    recommendations[
        ["company", "title", "final_score"]
    ]
)

                                 company                        title  \
461  INTELLIGENT SYSTEMS PRIVATE LIMITED  full stack developer intern   
214                            Optimasys    python development intern   
246                            Optimasys    python development intern   
735                            Optimasys    python development intern   
68                             Optimasys    python development intern   
138                            Optimasys    python development intern   
442                            Optimasys      python developer intern   
968                            Optimasys      python developer intern   
607                            Optimasys      python developer intern   
873                            Optimasys        python backend intern   

     final_score  
461     0.483333  
214     0.462654  
246     0.462654  
735     0.462654  
68      0.462654  
138     0.462654  
442     0.462184  
968     0.462184  
607     0.462184  
873   

In [15]:
from collections import Counter

def get_skill_gaps(student, top_n=10):

    recommendations = recommend_internships(
        student,
        top_n=top_n
    )

    student_skills = set(student["skills"])

    skill_counter = Counter()

    for _, internship in recommendations.iterrows():

        missing_skills = set(internship["skills"]) - student_skills

        for skill in missing_skills:
            skill_counter[skill] += 1

    skill_gaps = pd.DataFrame(
        skill_counter.items(),
        columns=["skill", "internship_count"]
    )

    if skill_gaps.empty:
        return skill_gaps

    skill_gaps["frequency"] = (
        skill_gaps["internship_count"] / len(recommendations)
    )

    skill_gaps = skill_gaps.sort_values(
        ["internship_count", "skill"],
        ascending=[False, True]
    ).reset_index(drop=True)

    return skill_gaps

In [16]:
def get_skill_recommendations(student, top_n=10):

    recommendations = recommend_internships(
        student,
        top_n=top_n
    )

    student_skills = set(student["skills"])

    skill_data = {}

    for _, internship in recommendations.iterrows():

        missing_skills = set(internship["skills"]) - student_skills

        for skill in missing_skills:

            if skill not in skill_data:
                skill_data[skill] = {
                    "internship_count": 0,
                    "importance_score": 0.0
                }

            skill_data[skill]["internship_count"] += 1
            skill_data[skill]["importance_score"] += internship["final_score"]

    skill_recommendations = pd.DataFrame(
        [
            {
                "skill": skill,
                "internship_count": data["internship_count"],
                "frequency": data["internship_count"] / len(recommendations),
                "importance_score": data["importance_score"]
            }
            for skill, data in skill_data.items()
        ]
    )

    if skill_recommendations.empty:
        return skill_recommendations

    # Normalize importance score
    max_importance = skill_recommendations["importance_score"].max()

    if max_importance > 0:
        skill_recommendations["importance_score"] = (
            skill_recommendations["importance_score"] / max_importance
        )

    # Final skill priority
    skill_recommendations["priority_score"] = (
        0.6 * skill_recommendations["frequency"]
        + 0.4 * skill_recommendations["importance_score"]
    )

    skill_recommendations = skill_recommendations.sort_values(
        "priority_score",
        ascending=False
    ).reset_index(drop=True)

    return skill_recommendations


def add_skill_priority(skill_recommendations):
    if skill_recommendations.empty:
        return skill_recommendations

    skill_recommendations["priority"] = skill_recommendations[
        "priority_score"
    ].apply(
        lambda score:
            "High" if score >= 0.50
            else "Medium" if score >= 0.20
            else "Low"
    )

    return skill_recommendations


def get_frontend_skill_recommendations(student, top_n=10):

    skill_recommendations = get_skill_recommendations(
        student,
        top_n
    )

    skill_recommendations = add_skill_priority(
        skill_recommendations
    )

    if skill_recommendations.empty:
        # Return an empty DataFrame with the expected columns if no recommendations
        return pd.DataFrame(columns=["skill", "priority", "internship_count"])

    result = skill_recommendations[
        [
            "skill",
            "priority",
            "internship_count"
        ]
    ].head(5).copy()

    return result

In [17]:
def get_recommendation_result(student, top_n=10):

    recommendations = recommend_internships(
        student,
        top_n=top_n
    )

    skill_recommendations = get_frontend_skill_recommendations(
        student,
        top_n=top_n
    )

    recommendation_result = recommendations[
        [
            "company",
            "title",
            "description",
            "location",
            "min_cgpa",
            "duration_months",
            "stipend",
            "positions_available",
            "application_deadline",
            "skills",
            "eligible_branches",
            "final_score"
        ]
    ].copy()

    recommendation_result = recommendation_result.rename(
        columns={
            "final_score": "match_score"
        }
    )

    return {
        "recommendations": recommendation_result.to_dict(
            orient="records"
        ),
        "skill_recommendations": skill_recommendations.to_dict(
            orient="records"
        )
    }

In [19]:
print("Please enter the student's details:")

branch_input = input("Branch (e.g., cse, it): ").strip().lower()
cgpa_input = float(input("CGPA (e.g., 8.5): ").strip())
skills_input = [s.strip().lower() for s in input("Skills (comma-separated, e.g., python, machine learning): ").split(',') if s.strip()]
interests_input = [i.strip().lower() for i in input("Interests (comma-separated, e.g., artificial intelligence, data science): ").split(',') if i.strip()]
career_goal_input = input("Career Goal (e.g., data scientist): ").strip().lower()
preferred_location_input = input("Preferred Location (e.g., remote, bangalore): ").strip().lower()
preferred_duration_input = int(input("Preferred Duration in months (e.g., 6): ").strip())
minimum_stipend_input = float(input("Minimum Stipend (e.g., 30000): ").strip())

test_student_profile = {
    "branch": branch_input,
    "cgpa": cgpa_input,
    "skills": skills_input,
    "interests": interests_input,
    "career_goal": career_goal_input,
    "preferred_location": preferred_location_input,
    "preferred_duration": preferred_duration_input,
    "minimum_stipend": minimum_stipend_input
}

print("\nTest Student Profile:", test_student_profile)

test_result = get_recommendation_result(
    test_student_profile,
    top_n=10
)

print("\nInternship Recommendations for Test Student:")
if test_result["recommendations"]:
    for rec in test_result["recommendations"]:
        print(f"  Company: {rec['company']}, Title: {rec['title']}, Match Score: {rec['match_score']*100:.2f}%")
else:
    print("  No internship recommendations found.")

print("\nSkill Recommendations for Test Student (Top 5):")
if test_result["skill_recommendations"]:
    for skill_rec in test_result["skill_recommendations"]:
        print(f"  Skill: {skill_rec['skill']}, Priority: {skill_rec['priority']}, Internship Count: {skill_rec['internship_count']}")
else:
    print("  No skill recommendations found.")

Please enter the student's details:
Branch (e.g., cse, it): it
CGPA (e.g., 8.5): 7
Skills (comma-separated, e.g., python, machine learning): python, mysql, flask, java, c
Interests (comma-separated, e.g., artificial intelligence, data science): artificial intelligence
Career Goal (e.g., data scientist): data scientist
Preferred Location (e.g., remote, bangalore): delhi
Preferred Duration in months (e.g., 6): 6
Minimum Stipend (e.g., 30000): 2500

Test Student Profile: {'branch': 'it', 'cgpa': 7.0, 'skills': ['python', 'mysql', 'flask', 'java', 'c'], 'interests': ['artificial intelligence'], 'career_goal': 'data scientist', 'preferred_location': 'delhi', 'preferred_duration': 6, 'minimum_stipend': 2500.0}

Internship Recommendations for Test Student:
  Company: Optimasys, Title: python development intern, Match Score: 39.00%
  Company: Optimasys, Title: python development intern, Match Score: 39.00%
  Company: Optimasys, Title: python development intern, Match Score: 39.00%
  Company: O